In [3]:
import pandas as pd
import pickle
import numpy as np
from tqdm import tqdm

In [15]:
df_snippet_score = pd.read_csv('./Result/llm_result/snippet_scores_list.csv')
df_snippet_score

,company_name,date,item_1a_scores,item_7_scores,total_scores
0,AAPL,2021-10-29,"[6, 3, 3, 3, 5, 7, 3, 2, 1, 3, 4, 2, 4, 2, 5, ...","[3, 3, 3, 7, 3, 5, 7, 2, 3]","[6, 3, 3, 3, 5, 7, 3, 2, 1, 3, 4, 2, 4, 2, 5, ..."
1,AAPL,2022-10-28,"[8, 3, 2, 3, 7, 9, 3, 0, 5, 5, 3, 7, 4, 3, 6, ...","[3, 2, 3, 4, 3, 6, 6, 2]","[8, 3, 2, 3, 7, 9, 3, 0, 5, 5, 3, 7, 4, 3, 6, ..."
2,AAPL,2023-11-03,"[4, 6]",[],"[4, 6]"
3,AAPL,2024-11-01,"[6, 1]",[],"[6, 1]"
4,ABBV,2021-02-19,"[7, 3, 4, 3, 2, 2, 3, 1]",[3],"[7, 3, 4, 3, 2, 2, 3, 1, 3]"
...,...,...,...,...,...
1723,ZBRA,2024-02-15,"[2, 2, 1, 6, 2, 5, 5, 5, 1]",[1],"[2, 2, 1, 6, 2, 5, 5, 5, 1, 1]"
1724,ZTS,2021-02-16,"[10, 6, 9, 7, 9, 8, 6, 4, 6, 4, 5, 4, 6]","[7, 4, 5, 7, 4, 6]","[10, 6, 9, 7, 9, 8, 6, 4, 6, 4, 5, 4, 6, 7, 4,..."
1725,ZTS,2022-02-15,"[8, 7, 7, 7, 6, 6, 7, 3, 4, 4, 4]","[6, 4]","[8, 7, 7, 7, 6, 6, 7, 3, 4, 4, 4, 6, 4]"
1726,ZTS,2023-02-14,"[9, 8, 5, 5, 6, 7, 1, 4, 3, 1, 1, 2, 3, 1, 2, ...","[8, 5, 9, 3, 1]","[9, 8, 5, 5, 6, 7, 1, 4, 3, 1, 1, 2, 3, 1, 2, ..."


In [2]:
# Preprocessing Risk Scores
df_snippet_score = pd.read_csv('./Result/llm_result/snippet_score_item_count.csv')
df_firm_score = df_snippet_score.groupby(['company_name', 'date']).agg(risk_score=('item_snippets_score', 'sum')).reset_index()
df_firm_score = df_firm_score.rename(columns={'company_name': 'ticker'})
df_firm_score['date'] = pd.to_datetime(df_firm_score['date'], format='%Y-%m-%d')
df_firm_score['year'] = df_firm_score['date'].dt.year - 1
df_firm_score = df_firm_score[['year', 'ticker', 'risk_score']].reset_index(drop=True)

df_item_score_1a = df_snippet_score[df_snippet_score['item_label'] == 'item_1a'][['company_name', 'date', 'item_snippets_score']].reset_index(drop=True)
df_item_score_1a = df_item_score_1a.rename(columns={'item_snippets_score': 'risk_score'})
df_item_score_1a = df_item_score_1a.rename(columns={'company_name': 'ticker'})
df_item_score_1a['date'] = pd.to_datetime(df_item_score_1a['date'], format='%Y-%m-%d')
df_item_score_1a['year'] = df_item_score_1a['date'].dt.year - 1
df_item_score_1a = df_item_score_1a[['year', 'ticker', 'risk_score']].reset_index(drop=True)

df_item_score_7 = df_snippet_score[df_snippet_score['item_label'] == 'item_7'][['company_name', 'date', 'item_snippets_score']].reset_index(drop=True)
df_item_score_7 = df_item_score_7.rename(columns={'item_snippets_score': 'risk_score'})
df_item_score_7 = df_item_score_7.rename(columns={'company_name': 'ticker'})
df_item_score_7['date'] = pd.to_datetime(df_item_score_7['date'], format='%Y-%m-%d')
df_item_score_7['year'] = df_item_score_7['date'].dt.year - 1
df_item_score_7 = df_item_score_7[['year', 'ticker', 'risk_score']].reset_index(drop=True)

# All items scores
df_firm_score.to_csv('./Data/Fama_French/risk_scores.csv', index=False)

# Items 1a scores
df_item_score_1a.to_csv('./Data/Fama_French/risk_scores_1a.csv', index=False)

# Items 7 scores
df_item_score_7.to_csv('./Data/Fama_French/risk_scores_7.csv', index=False)

print('*** Firm Score ***\n', df_firm_score, end='\n\n')
print('*** Item 1a Score ***\n', df_item_score_1a, end='\n\n')
print('*** Item 7 Score ***\n', df_item_score_7, end='\n\n')

*** Firm Score ***
       year ticker  risk_score
0     2020      A          46
1     2021      A          55
2     2023      A          22
3     2020   AAPL          94
4     2021   AAPL         114
...    ...    ...         ...
1704  2023   ZBRA          30
1705  2020    ZTS         117
1706  2021    ZTS          73
1707  2022    ZTS          89
1708  2023    ZTS          22

[1709 rows x 3 columns]

*** Item 1a Score ***
       year ticker  risk_score
0     2020      A          46
1     2021      A          55
2     2023      A          22
3     2020   AAPL          58
4     2021   AAPL          85
...    ...    ...         ...
1697  2023   ZBRA          29
1698  2020    ZTS          84
1699  2021    ZTS          63
1700  2022    ZTS          63
1701  2023    ZTS          18

[1702 rows x 3 columns]

*** Item 7 Score ***
       year ticker  risk_score
0     2020   AAPL          36
1     2021   AAPL          29
2     2020   ABBV           3
3     2021   ABBV           7
4     2022   

In [3]:
with open('./Data/price_cache.pkl', 'rb') as f:
    company_price_cache = pickle.load(f)

for key in company_price_cache.keys():
    company_price_cache[key]['return'] = company_price_cache[key]['Close'].pct_change()

def get_return(ticker, date):
    try:
        daily_return = company_price_cache[ticker].loc[date]['return'].values[0]
        return daily_return
    except Exception as e:
        # print(f"Error retrieving price for {ticker} on {date}: {e}")
        return None

In [4]:
tickers = sorted(set(df_firm_score['ticker'].to_list()))

start_date = '2020-01-01'
end_date = '2025-06-30'

dates = pd.to_datetime(pd.date_range(start_date, end_date, freq='B')) # 'B' for business day
multi_index = pd.MultiIndex.from_product([dates, tickers], names=['date', 'ticker'])
returns_df = pd.DataFrame(index=multi_index)

return_list = []
for row in tqdm(returns_df.index):
    ticker = row[1]
    date = row[0]
    return_list.append(get_return(ticker, date))

returns_df['return'] = return_list
returns_df = returns_df.dropna().reset_index()
returns_df.to_csv('./Data/Fama_French/stock_daily_data.csv', index=False)

100%|██████████| 658206/658206 [00:40<00:00, 16121.19it/s]
